# Notebook 01 — Grounding DINO 기초

> 텍스트 프롬프트로 임의의 객체를 감지하는 오픈 어휘 Detection

## 이 노트북에서 다루는 것

1. Grounding DINO 작동 원리 — 왜 YOLO와 다른가
2. 모델 가중치 다운로드
3. 텍스트 프롬프트 → 바운딩 박스 추출
4. 임계값(threshold) 튜닝 실험
5. YOLO vs Grounding DINO 비교

---

## 핵심 개념

```
YOLO:            이미지 → 고정 80개 클래스 중 하나로 분류
Grounding DINO:  이미지 + 텍스트 → 텍스트와 시각적으로 정렬된 영역 추출
```

**DINO** (자기지도 Vision Transformer) + **BERT** (텍스트 인코더) 결합 →  
텍스트와 이미지 패치를 같은 임베딩 공간에 정렬 → 제로샷 감지 가능

In [ ]:
# 한글 폰트 설정
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False
print('✅ 폰트 설정 완료')

## 1. 모델 가중치 다운로드

In [ ]:
import os
import urllib.request
from pathlib import Path

# 저장 경로
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(exist_ok=True)

WEIGHTS_PATH = MODEL_DIR / 'groundingdino_swint_ogc.pth'
CONFIG_PATH  = MODEL_DIR / 'GroundingDINO_SwinT_OGC.py'

WEIGHTS_URL = 'https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth'
CONFIG_URL  = 'https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/groundingdino/config/GroundingDINO_SwinT_OGC.py'

def download_if_missing(url, path):
    if path.exists():
        print(f'✅ 이미 존재: {path.name}')
        return
    print(f'⬇️  다운로드 중: {path.name}')
    urllib.request.urlretrieve(url, path)
    print(f'✅ 완료: {path.name}')

download_if_missing(WEIGHTS_URL, WEIGHTS_PATH)
download_if_missing(CONFIG_URL,  CONFIG_PATH)

print(f'\n모델 크기: {WEIGHTS_PATH.stat().st_size / 1e6:.1f} MB')

## 2. 모델 로드

In [ ]:
import torch
from groundingdino.util.inference import load_model

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

model = load_model(str(CONFIG_PATH), str(WEIGHTS_PATH))
model = model.to(DEVICE)
model.eval()
print('✅ Grounding DINO 모델 로드 완료')

## 3. 테스트 이미지 준비

URL에서 샘플 이미지를 다운로드합니다.

In [ ]:
import urllib.request
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

IMAGE_DIR = Path('../data/test_images')
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# 공개 샘플 이미지 (거리 장면)
SAMPLE_IMAGES = {
    'street.jpg': 'https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/640px-Cute_dog.jpg',
    'office.jpg':  'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/640px-Cat03.jpg',
}

for fname, url in SAMPLE_IMAGES.items():
    fpath = IMAGE_DIR / fname
    if not fpath.exists():
        urllib.request.urlretrieve(url, fpath)
        print(f'⬇️  다운로드: {fname}')
    else:
        print(f'✅ 존재: {fname}')

# 첫 번째 이미지 미리보기
img = Image.open(IMAGE_DIR / 'street.jpg').convert('RGB')
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.axis('off')
plt.title('테스트 이미지')
plt.tight_layout()
plt.show()
print(f'이미지 크기: {img.size}')

## 4. 텍스트 프롬프트 → 바운딩 박스 추출

### 프롬프트 형식
- 단일: `"dog"`
- 복수: `"dog . cat . person"` (점으로 구분)
- 자연어: `"a small brown dog"`

In [ ]:
from groundingdino.util.inference import load_image, predict, annotate
import cv2

def detect_and_visualize(image_path, prompt, box_threshold=0.3, text_threshold=0.25, title=''):
    """Grounding DINO 추론 + 시각화"""
    image_source, image_tensor = load_image(str(image_path))

    boxes, logits, phrases = predict(
        model=model,
        image=image_tensor,
        caption=prompt,
        box_threshold=box_threshold,
        text_threshold=text_threshold,
        device=DEVICE,
    )

    annotated = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(10, 7))
    plt.imshow(annotated_rgb)
    plt.axis('off')
    plt.title(f'{title}\n프롬프트: "{prompt}" | 감지: {len(boxes)}개', fontsize=12)
    plt.tight_layout()
    plt.show()

    print(f'\n[감지 결과]')
    for i, (box, logit, phrase) in enumerate(zip(boxes, logits, phrases)):
        print(f'  {i+1}. {phrase:20s} | 신뢰도: {logit:.4f} | 박스: {box.tolist()}')

    return boxes, logits, phrases

# 기본 감지
boxes, logits, phrases = detect_and_visualize(
    IMAGE_DIR / 'street.jpg',
    prompt='dog',
    title='단일 텍스트 프롬프트'
)

## 5. 임계값(Threshold) 튜닝 실험

- **box_threshold**: 낮을수록 더 많이 감지 (FP 증가)
- **text_threshold**: 텍스트-이미지 정렬 신뢰도

In [ ]:
import pandas as pd

image_path = IMAGE_DIR / 'street.jpg'
prompt = 'dog'

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]
results = []

for thresh in thresholds:
    image_source, image_tensor = load_image(str(image_path))
    boxes, logits, phrases = predict(
        model=model,
        image=image_tensor,
        caption=prompt,
        box_threshold=thresh,
        text_threshold=thresh * 0.8,
        device=DEVICE,
    )
    results.append({
        'box_threshold': thresh,
        '감지 수': len(boxes),
        '평균 신뢰도': round(float(logits.mean()), 4) if len(logits) > 0 else 0.0,
        '최고 신뢰도': round(float(logits.max()), 4) if len(logits) > 0 else 0.0,
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df['box_threshold'], df['감지 수'], 'o-', color='steelblue')
axes[0].set_xlabel('box_threshold')
axes[0].set_ylabel('감지된 객체 수')
axes[0].set_title('임계값 vs 감지 수')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df['box_threshold'], df['평균 신뢰도'], 's-', color='tomato', label='평균')
axes[1].plot(df['box_threshold'], df['최고 신뢰도'], '^--', color='orange', label='최고')
axes[1].set_xlabel('box_threshold')
axes[1].set_ylabel('신뢰도')
axes[1].set_title('임계값 vs 신뢰도')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../output/01_threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 저장: output/01_threshold_sweep.png')

## 6. 복수 프롬프트 — 여러 객체 동시 감지

In [ ]:
# 점(.) 으로 구분해서 여러 클래스 동시 감지
detect_and_visualize(
    IMAGE_DIR / 'street.jpg',
    prompt='dog . grass . background',
    box_threshold=0.3,
    title='복수 프롬프트 감지'
)

## 7. YOLO vs Grounding DINO 비교 분석

In [ ]:
import matplotlib.patches as mpatches

comparison = {
    '항목': ['감지 방식', '클래스 수', '새 클래스 추가', '추론 속도', '정확도 (COCO)', '사용 편의성'],
    'YOLOv8': [
        '학습된 클래스 분류',
        '80개 고정',
        '재학습 필요',
        '빠름 (~50ms)',
        'mAP 53.9',
        '높음 (설정 간단)',
    ],
    'Grounding DINO': [
        '텍스트-이미지 정렬',
        '무제한 (자연어)',
        '프롬프트만 수정',
        '느림 (~200ms)',
        'AP 48.4 (제로샷)',
        '높음 (프롬프트 작성)',
    ],
}

df_cmp = pd.DataFrame(comparison).set_index('항목')
print('=== YOLO vs Grounding DINO ===')
print(df_cmp.to_string())

print('\n[결론]')
print('- 속도 우선, 고정 클래스 환경  → YOLOv8')
print('- 유연성 우선, 새로운 도메인    → Grounding DINO')
print('- 최선의 결합: DINO로 탐색 → YOLO로 특화 학습')

## 8. 속도 벤치마크

In [ ]:
import time

image_source, image_tensor = load_image(str(IMAGE_DIR / 'street.jpg'))
prompt = 'dog . person . car'
N = 10

# 워밍업
predict(model=model, image=image_tensor, caption=prompt,
        box_threshold=0.3, text_threshold=0.25, device=DEVICE)

# 벤치마크
times = []
for _ in range(N):
    start = time.time()
    predict(model=model, image=image_tensor, caption=prompt,
            box_threshold=0.3, text_threshold=0.25, device=DEVICE)
    times.append((time.time() - start) * 1000)

print(f'추론 속도 (n={N})')
print(f'  평균: {sum(times)/len(times):.1f} ms')
print(f'  최소: {min(times):.1f} ms')
print(f'  최대: {max(times):.1f} ms')
print(f'  FPS:  {1000/(sum(times)/len(times)):.1f}')

plt.figure(figsize=(8, 4))
plt.plot(times, 'o-', color='steelblue')
plt.axhline(sum(times)/len(times), color='tomato', linestyle='--', label=f'평균 {sum(times)/len(times):.1f}ms')
plt.xlabel('반복 횟수')
plt.ylabel('추론 시간 (ms)')
plt.title('Grounding DINO 추론 속도')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../output/01_speed_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 저장: output/01_speed_benchmark.png')

## 9. 결과 요약

In [ ]:
print('=' * 50)
print('Notebook 01 — Grounding DINO 기초 완료')
print('=' * 50)
print()
print('[학습한 것]')
print('  1. Grounding DINO: DINO + BERT 결합으로 텍스트-이미지 정렬')
print('  2. 텍스트 프롬프트로 임의 객체 제로샷 감지')
print('  3. box_threshold / text_threshold 튜닝 효과')
print('  4. YOLO(속도) vs DINO(유연성) 트레이드오프')
print()
print('[저장된 결과물]')
print('  - output/01_threshold_sweep.png')
print('  - output/01_speed_benchmark.png')
print()
print('[다음 노트북]')
print('  02_sam2_integration.ipynb')
print('  → DINO 박스를 SAM2 프롬프트로 연결, 픽셀 마스크 생성')